# Step 2: Filter notebook

In [ ]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt
import rasterio
import numpy as np

import os

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm

# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)

## Imports

#### Import des segments

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input/attributs'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

save_filtered_attributes = True

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):

    # Clean geometries first
    gdf["geometry"] = gdf["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
    print("Geometries cleaned")
    
    if save_filtered_attributes:
        if row['save_format'] == 'parquet':
            if not os.path.exists(f'{output_step2_path}/gpkg_attributs'):
                os.makedirs(f'{output_step2_path}/gpkg_attributs')
            gdf.to_parquet(f"{output_step2_path}/parquet_attributs/{attribute}.parquet")
            print(f"Filtered data saved for attribute: {attribute} in format: {row['save_format']}")
        elif row['save_format'] == 'gpkg':
            if not os.path.exists(f'{output_step2_path}/parquet_attributs'):
                os.makedirs(f'{output_step2_path}/parquet_attributs')
            gdf.to_file(f"{output_step2_path}/gpkg_attributs/{attribute}.gpkg", driver="GPKG")
            print(f"Filtered data saved for attribute: {attribute} in format: {row['save_format']}")
        else:
            if not os.path.exists(f'{output_step2_path}/csv_attributs'):
                os.makedirs(f'{output_step2_path}/csv_attributs')
            gdf.to_csv(f"{output_step2_path}/csv_attributs/{attribute}.csv", index=False)
            print(f"Warning: Unknown save format {row['save_format']} for attribute {attribute}. Data saved as csv.")
    else:
        print("Note : Save option is disabled.")

#### Import des attributs 

In [ ]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs_info.xlsx", sheet_name="attributs_info")

In [ ]:
attributs_info

**Attribut Accidents**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'accident'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.ANNEE > 2020, 'filtered'] = 1
gdf.loc[gdf.CONSEQ.isin(['Avec blessés graves', 'Avec tués']), 'filtered'] = 1
gdf.loc[gdf.VELOS == 1, 'filtered'] = 1
gdf.loc[gdf.VAE_25 == 1, 'filtered'] = 1
gdf.loc[gdf.VAE_45 == 1, 'filtered'] = 1
gdf.loc[gdf.PIETONS == 1, 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Arbres isolés** (groupe Végétation)

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'arbre_isole'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)


**Attribut Espaces verts** (groupe Végétation)

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'espace_vert'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.REVETEMENT.isin(['Arbustes', 'Terre', 'Gazon','Grille gazon', 'Prairie', 'Plates-bandes']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Vitesse** (groupe Traffic)

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'vitesse'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_ZONE == 0, 'filtered'] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Zone pietonne** (groupe Traffic)

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_pietonne'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_ZONE.isin(['Zone piétonne', 'Zone de rencontre (20)', ]), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Zone apaisée** (groupe Traffic)

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_apaisee'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_LIMIT.isin(['Zone 30 km/h', 'Prescription 30 km/h', 'Prescription 20 km/h']), 'filtered'] = 1 

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Largeur trottoirs**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'ratio_trottoir'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0


# Filter criterion
###----change here----------------------
gdf.loc[gdf.OBJET.isin(['Trottoir']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Eau**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eau'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.ETAT.isin(['A ciel ouvert']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Rez Actifs**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'rez_actif'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'rez_actif'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
#gdf.loc[gdf['BRANCHE'].str.contains('commerce de détail|détail|écoles|commerces|supermarchés|restaurants|banques|enseignement', case=False, na=False), 'filtered'] = 1
keywords = ['commerce de détail', 'droguerie', 'enseignement', 'Hôpitaux', 'Hôtels', "Maisons", 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Protection civile', 'Restaurants', 'Salons', 'Écoles'  ]
gdf.loc[gdf.BRANCHE.dropna().str.lower().apply(lambda val: any(kw.lower() in val for kw in keywords)), 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Bruit**

In [ ]:

# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'bruit'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.CAT_J > 2, 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Proximité TP**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'tp'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Stationnement Genants**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'stationnement_genant'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf[gdf.geometry.notna()].copy()  # Remove rows with None geometries
gdf = gdf[gdf.geometry.is_valid].copy()  # Keep only valid geometries
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Proximité Aménités**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'amenite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
#keep if the column "BRANCHE" contains one of the following keywords (case insensitive)
keywords = ['commerce de détail', 'droguerie', 'enseignement', 'Hôpitaux', 'Hôtels', "Maisons", 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Protection civile', 'Restaurants', 'Salons', 'Écoles'  ]
gdf.loc[gdf.BRANCHE.dropna().str.lower().apply(lambda val: any(kw.lower() in val for kw in keywords)), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut espaces ouverts**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'espaces_ouverts'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE == 'ESPACE PUBLIC', 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Confort thermique**

In [ ]:
# Initialize

attribute = 'temperature'
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
raster_path = (f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")

with rasterio.open(raster_path) as src:
    # Read the raster data
    raster_data = src.read(1)  # Read first band
    
    # Define NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    
    # Valid mask: remove NoData
    valid_mask = (raster_data != nodata_val)
    
    # Get coordinates of valid points
    rows, cols = np.where(valid_mask)
    
    # Sample every 10th point
    #rows, cols = rows[::10], cols[::10]
    
    # Get coordinates in map units
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    temp_values = raster_data[rows, cols]
    
    # Create GeoDataFrame directly with filtered points
    gdf = gpd.GeoDataFrame({
        'temperature': temp_values,
        'filtered': 1,  # All points are filtered since we pre-filtered the data
        'geometry': [Point(x, y) for x, y in zip(xs, ys)]
    }, geometry='geometry')
    
    # Set CRS and convert to target CRS
    gdf = gdf.set_crs(src.crs)
    gdf = gdf.to_crs(target_crs)

print(f"Processing attribute: {attribute}")
print("\nValue counts (first 10 most common values):")
print(gdf['temperature'].value_counts().head(10))
print("\nDescriptive statistics:")
print(gdf['temperature'].describe())

# Save
print(f"Saving {attribute}: ca take up to 3mn")
save(save_filtered_attributes, row, gdf, attribute)

In [ ]:
#temp_parquet = gpd.read_parquet('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/parquet_attributs/temperature.parquet')
#temp_parquet.to_file('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs/temperature.gpkg', driver="GPKG")


**Largeur trottoir**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'largeur_trottoir'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.Largeur.isin(['Très large', 'Large','Moyen']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Topographie**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'topographie'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)